# 8.2. 文本预处理

将文本作为字符串加载到内存中。

将字符串拆分为词元（如单词和字符）。

建立一个词表，将拆分的词元映射到数字索引。

将文本转换为数字索引序列，方便模型操作。

In [21]:
import collections
import re
from d2l import torch as d2l
DATA_DIR = '/home/beichen/jiaxinl/data/timemachine.txt'

In [22]:
#@save
d2l.DATA_HUB['time_machine'] = (d2l.DATA_URL + 'timemachine.txt',
                                '090b5e7e70c295757f55df93cb0a180b9691891a')

# 读取文本、清洗文本
def read_time_machine():  #@save
    """将时间机器数据集加载到文本行的列表中"""
    path = DATA_DIR
    with open(path, 'r') as f:
        lines = f.readlines() # f.readlines()：读取文件的所有行，返回一个列表。
    return [re.sub('[^A-Za-z]+', ' ', line).strip().lower() for line in lines]
# 如果是中文的话怎么弄
lines = read_time_machine()
print(f'# 文本总行数: {len(lines)}')
print(lines[0])
print(lines[10])

# 文本总行数: 3221
the time machine by h g wells
twinkled and his usually pale face was flushed and animated the


- `re.sub()`：使用正则表达式替换文本
- `[^A-Za-z]+`:匹配一个或多个非英文字母
- `.strip()`:去掉字符串首尾的空格
- `.lower()`:全部转换为小写

## 8.2.2. 词元化

In [3]:
def tokenize(lines, token='word'):  #@save
    """将文本行拆分为单词或字符词元"""
    if token == 'word': # 单词
        return [line.split() for line in lines]
    elif token == 'char': # 字符
        return [list(line) for line in lines]
    else:
        print('错误：未知词元类型：' + token)
# 两种拆分方式，模型可以选择不同的文本处理粒度
# 单词级：["time", "machine"]，字符级：["t", "i", "m", "e", ...]
tokens = tokenize(lines)
for i in range(11):
    print(tokens[i])

['the', 'time', 'machine', 'by', 'h', 'g', 'wells']
[]
[]
[]
[]
['i']
[]
[]
['the', 'time', 'traveller', 'for', 'so', 'it', 'will', 'be', 'convenient', 'to', 'speak', 'of', 'him']
['was', 'expounding', 'a', 'recondite', 'matter', 'to', 'us', 'his', 'grey', 'eyes', 'shone', 'and']
['twinkled', 'and', 'his', 'usually', 'pale', 'face', 'was', 'flushed', 'and', 'animated', 'the']


- `line.split()`:按照空白字符拆分字符串
```python
        line = "the time machine"
        line.split()
        结果：["the", "time", "machine"]
```

- `list(line)`:会把字符串拆成一个个字符
```python
        list("time")
        结果：["t", "i", "m", "e"]
```

## 8.2.3. 词表

词元的类型是字符串，而模型需要的输入是数字，因此这种类型不方便模型使用。 现在，让我们构建一个字典，通常也叫做词表（vocabulary）， 用来将字符串类型的词元映射到从0
开始的数字索引中。

- `tokens`；传入要建立词表的文本
- `reserved_tokens`：指定必须保留的特殊词元

In [4]:
class Vocab:  #@save
    """文本词表"""
    def __init__(self, tokens=None, min_freq=0, reserved_tokens=None):
        if tokens is None:
            tokens = []  # 如果创建词表时没有传入tokens,就使用空列表
        if reserved_tokens is None:
            reserved_tokens = []
        # 按出现频率排序
        counter = count_corpus(tokens)# 统计每个词元出现多少次。
        # 按照出现频率从高到低排序。
        self._token_freqs = sorted(
            counter.items(), # 取出词元和频率组成的键值对。
            key=lambda x: x[1],# 按照每个键值对的第二个元素排序，也就是频率
            reverse=True # 从大到小排序。
            )
        # 未知词元的索引为0
        self.idx_to_token = ['<unk>'] + reserved_tokens
        self.token_to_idx = {
            token: idx
            for idx, token in enumerate(self.idx_to_token)}
        for token, freq in self._token_freqs:
            if freq < min_freq:
                break
            if token not in self.token_to_idx:
                self.idx_to_token.append(token)
                self.token_to_idx[token] = len(self.idx_to_token) - 1

    def __len__(self):
        return len(self.idx_to_token)

    # 把词元转换成数字
    def __getitem__(self, tokens):
        if not isinstance(tokens, (list, tuple)):
            return self.token_to_idx.get(tokens, self.unk)
        return [self.__getitem__(token) for token in tokens]
    # 把数字转换回词元
    def to_tokens(self, indices):
        if not isinstance(indices, (list, tuple)):
            return self.idx_to_token[indices]
        return [self.idx_to_token[index] for index in indices]

    @property
    def unk(self):  # 未知词元的索引为0
        return 0

    @property
    def token_freqs(self):
        return self._token_freqs

def count_corpus(tokens):  #@save
    """统计词元的频率"""
    # 这里的tokens是1D列表或2D列表
    if len(tokens) == 0 or isinstance(tokens[0], list):
        # 将词元列表展平成一个列表
        tokens = [token for line in tokens for token in line]
    return collections.Counter(tokens)

- `isinstance(变量, 类型)`:判断变量是不是指定的类型。

- `counter = count_corpus(tokens)`:
结果：
```python
{
    "the": 2,
    "time": 2,
    "machine": 1,
    "by": 1
}
```

In [5]:
vocab = Vocab(tokens)
print(list(vocab.token_to_idx.items())[:10])

[('<unk>', 0), ('the', 1), ('i', 2), ('and', 3), ('of', 4), ('a', 5), ('to', 6), ('was', 7), ('in', 8), ('that', 9)]


In [6]:
# 分别取出第 0 行和第 10 行文本，然后看看它们转换成数字编号之后是什么样子。
for i in [0, 10]:
    print('文本:', tokens[i])
    print('索引:', vocab[tokens[i]]) # 调用def __getitem__(self, tokens):

文本: ['the', 'time', 'machine', 'by', 'h', 'g', 'wells']
索引: [1, 19, 50, 40, 2183, 2184, 400]
文本: ['twinkled', 'and', 'his', 'usually', 'pale', 'face', 'was', 'flushed', 'and', 'animated', 'the']
索引: [2186, 3, 25, 1044, 362, 113, 7, 1421, 3, 1045, 1]


In [7]:
def load_corpus_time_machine(max_tokens=-1):  #@save
    """返回时光机器数据集的词元索引列表和词表"""
    lines = read_time_machine()
    tokens = tokenize(lines, 'char')
    vocab = Vocab(tokens) # 创建一个词表对象。
    # 因为时光机器数据集中的每个文本行不一定是一个句子或一个段落，
    # 所以将所有文本行展平到一个列表中
    corpus = [vocab[token] for line in tokens for token in line]
    if max_tokens > 0: # 限制数据长度
        corpus = corpus[:max_tokens]
    return corpus, vocab

corpus, vocab = load_corpus_time_machine()
len(corpus), len(vocab)
#corpus:整本书转换得到的数字序列
#vocab;字符和数字之间的映射表

(170580, 28)

# 8.3. 语言模型和数据集

In [23]:
import random
import torch
from d2l import torch as d2l
tokens = d2l.tokenize(read_time_machine())
# 因为每个文本行不一定是一个句子或一个段落，因此我们把所有文本行拼接到一起
corpus = [token for line in tokens for token in line]
vocab = d2l.Vocab(corpus)
vocab.token_freqs[:10]

[('the', 2261),
 ('i', 1267),
 ('and', 1245),
 ('of', 1155),
 ('a', 816),
 ('to', 695),
 ('was', 552),
 ('in', 541),
 ('that', 443),
 ('my', 440)]